# Lecture 21 - Multiple Linear Regression & Residuals

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
plt.style.use('fivethirtyeight')

%matplotlib inline

In [ ]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.offline as po
po.init_notebook_mode()

In [ ]:
from scipy import optimize
import functools
import math

## Lecture Outline

- Multiple Linear Regression
- Polynomial Regression: Fitting Non-Linear Data  
  - Quadratic Equation
- Residuals
  - Interpretation and Plots  
  - Multiple Demos


## This function using scipy.optimize.minimize function 
- #### Used for Demonstration purpose

In [ ]:
# from scipy import optimize
# import functools
# import math

# You don't need to understand the details
# This function using scipy.optimize.minimize function 

def minimize(f, start=None, smooth=False, log=None, array=False, **vargs):
    if start is None:
        assert not array, "Please pass starting values explicitly when array=True"
        arg_count = f.__code__.co_argcount
        assert arg_count > 0, "Please pass starting values explicitly for variadic functions"
        start = [0] * arg_count
    if not hasattr(start, '__len__'):
        start = [start]

    if array:
        objective = f
    else:
        @functools.wraps(f)
        def objective(args):
            return f(*args)

    if not smooth and 'method' not in vargs:
        vargs['method'] = 'Powell'
    result = optimize.minimize(objective, start, **vargs)
    if log is not None:
        log(result)
    if len(start) == 1:
        return result.x.item(0)
    else:
        return result.x

## 📈 Multiple Linear Regression

- Models one dependent variable using two or more predictors.  
- Extends simple regression to capture combined effects of multiple inputs.  
- General form:  
  $
  y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_n x_n + \epsilon
  $


### Demo 1: Vehicle Data
- #### We can use multiple variables to help predict a single variable.

In [ ]:
hybrid = pd.read_csv('data/hybrid.csv')
hybrid.head(5)

In [ ]:
px.scatter_3d(
    hybrid, 
    x="mpg", y="acceleration", z="msrp",
    hover_name="vehicle", 
    color="class", 
    height=800
)

#### Suppose we use the model: 

$$ y = a * acc + b * mpg + c$$

In [ ]:
def hybrid_rmse(a, b, c):
    actual = hybrid["msrp"]
    acc = hybrid["acceleration"]
    mpg = hybrid["mpg"]
    predicted = a*acc + b*mpg + c
    mse = np.sqrt(np.mean((actual - predicted)**2))
    return mse

#### The optimizer iteratively adjusts a, b, and c until it finds the combination that minimizes RMSE.
- The minimize() wrapper then returns these optimized coefficients.

In [ ]:
a, b, c = minimize(hybrid_rmse)
[a, b, c]

In [ ]:
print(f"Error: {hybrid_rmse(a, b, c):,}")

### 🎯 In short

- `hybrid_rmse(a, b, c)` measures how wrong your model is.  
- `minimize()` finds the values of `a`, `b`, and `c` that make that error as small as possible.  
- The result gives your **best-fit linear model** predicting car prices (`msrp`) from **acceleration** and **mpg**.


In [ ]:
mpg_range = np.arange(10, 80)
acceleration_range = np.arange(5, 25)
predictions = pd.DataFrame(columns=["mpg", "acc", "pred"])
i = 0
for mpg in mpg_range:
    for acc in acceleration_range: 
        pred = a * acc + b * mpg + c
        predictions.loc[i] = [mpg, acc, pred]
        i = i+1

In [ ]:
fig = px.scatter_3d(
    hybrid, 
    x="mpg", y="acceleration", z="msrp",
    hover_name="vehicle", 
    color="class", 
    height=800
)
fig.add_surface(
    x = mpg_range, y = acceleration_range,
    z = predictions["pred"].to_numpy().reshape(len(mpg_range), len(acceleration_range)).T,
    showscale=False
)

## 🔹 Polynomial Regression: Fitting Non-Linear Data

- **Polynomial Regression** models non-linear relationships by adding higher-order terms (e.g., $x^2, x^3$) to a linear model.  
- It captures curved trends that simple linear regression cannot represent.  
- General form:  
  $
  y = \beta_0 + \beta_1 x + \beta_2 x^2 + \cdots + \beta_n x^n + \epsilon
  $
- Useful when data shows a smooth but non-linear pattern.


#### Demo 2: Ploynomial Regression 
- We could try to improve our predictions by defining a more complex equation: 

$$y = a * acc + b * mpg + c * acc^2 + d * mpg^2 + e$$

In [ ]:
def hybrid_better_rmse(a, b, c, d, e):
    actual = hybrid["msrp"]
    acc = hybrid["acceleration"]
    mpg = hybrid["mpg"]
    predicted = a*acc + b*mpg + c*acc**2 + d*mpg**2 + e
    mse = np.sqrt(np.mean((actual - predicted)**2))
    return mse

### 🧩 Purpose

- `hybrid_better_rmse(a, b, c, d, e)` calculates the **Root Mean Squared Error (RMSE)** for model evaluation.  
- It fits a **quadratic (non-linear) regression model** predicting car prices (`msrp`) from **acceleration** and **mpg**, including their squared terms.


In [ ]:
a, b, c, d, e = minimize(hybrid_better_rmse)
[a, b, c, d, e]

In [ ]:
print(f"Error: {hybrid_better_rmse(a,b,c,d,e):,}")

### 🎯 Interpretation

These optimized `a`, `b`, `c`, `d`, `e` define your **best-fit quadratic model**:

$
\text{Predicted MSRP} = a \cdot \text{acc} + b \cdot \text{mpg} + c \cdot \text{acc}^2 + d \cdot \text{mpg}^2 + e
$

It’s a **non-linear surface** relating acceleration and fuel efficiency to car price — usually with **lower RMSE** than the simple linear model.


### 3D scatter plot 
The code cell below visualizes a **3D scatter plot** of actual car data and overlays a **predicted quadratic surface** that represents the fitted model:

$
\text{Predicted MSRP} = a \cdot \text{acc} + b \cdot \text{mpg} + c \cdot \text{acc}^2 + d \cdot \text{mpg}^2 + e
$


In [ ]:
mpg_range = np.arange(10, 80)
acceleration_range = np.arange(5, 25)
predictions = pd.DataFrame(columns=["mpg", "acc", "pred"])
i = 0
for mpg in mpg_range:
    for acc in acceleration_range: 
        pred = a*acc + b*mpg + c*acc**2 + d*mpg**2 + e
        predictions.loc[i] = [mpg, acc, pred]
        i = i+1
        
fig = px.scatter_3d(
    hybrid, 
    x="mpg", y="acceleration", z="msrp",
    hover_name="vehicle", 
    color="class", 
    height=800
)
fig.add_surface(
    x = mpg_range, y = acceleration_range,
    z = predictions["pred"].to_numpy().reshape(len(mpg_range), len(acceleration_range)).T,
    showscale=False
)

### 🔍 Observations

- The 3D scatter shows actual car data, while the surface represents the quadratic model fit.  
- The surface captures non-linear trends between `mpg`, `acceleration`, and `msrp`.  
- Higher acceleration generally links to higher prices; higher mpg relates to lower prices.  
- The model fits well overall but some variation remains due to other factors.


## ✅ Please check Demo 3 by yourselves with your friends
### Demo 3: Another Nonlinear Regression Example

In [ ]:
shotput = pd.read_csv('data/shotput.csv')
shotput.head()

In [ ]:
shotput.plot.scatter(x='Weight Lifted', y='Shot Put Distance')

In [ ]:
def shotput_linear_rmse(any_slope, any_intercept):
    x = shotput['Weight Lifted']
    y = shotput['Shot Put Distance']
    estimate = any_slope*x + any_intercept
    return np.mean((y - estimate) ** 2) ** 0.5

In [ ]:
best_line = minimize(shotput_linear_rmse)
best_line

In [ ]:
weights = shotput.iloc[:,0]

In [ ]:
linear_fit = best_line[0]*weights + best_line[1]

shotput['Best Line'] = linear_fit

In [ ]:
plt.scatter(shotput['Weight Lifted'],shotput['Shot Put Distance'], label='Shot Put Distance')
plt.scatter(shotput['Weight Lifted'], shotput['Best Line'], label='Best Line')
plt.xlabel('Weight Lifted')
plt.ylabel('Shot Put Distance')
plt.legend()

### Quadratic Function 

$f(x) = ax^2 + bx + c$  

for constants $a$, $b$, and $c$.

In [ ]:
def shotput_quadratic_rmse(a, b, c):
    x = shotput['Weight Lifted']
    y = shotput['Shot Put Distance']
    estimate = a*(x**2) + b*x + c
    return np.mean((y - estimate) ** 2) ** 0.5

In [ ]:
best_quad = minimize(shotput_quadratic_rmse)
best_quad

In [ ]:
# x = weight lifted = 100 kg
# Then predicted shot put distance:

(-0.00104)*(100**2) + 0.2827*100 - 1.5318

In [ ]:
quad_fit = best_quad[0]*(weights**2) + best_quad[1]*weights + best_quad[2]

In [ ]:
shotput['Best Quadratic Curve'] = quad_fit

In [ ]:
plt.scatter(shotput['Weight Lifted'],shotput['Shot Put Distance'], label='Shot Put Distance')
plt.scatter(shotput['Weight Lifted'], shotput['Best Line'], label='Best Line')
plt.scatter(shotput['Weight Lifted'], shotput['Best Quadratic Curve'], label='Best Quad. Line')
plt.xlabel('Weight Lifted')
plt.ylabel('Shot Put Distance')
plt.legend()

---
# Residual
<br>

## Useful Functions for Regression line and Residual 

In [ ]:
def standard_units(x):
    "Convert any array of numbers to standard units."
    return (x - np.mean(x)) / np.std(x)

def correlation(df, x, y):
    """Computes the correlation between columns x and y"""
    x_su = standard_units(df[x])
    y_su = standard_units(df[y])
    return np.mean(x_su * y_su)

def slope(df, x, y):
    """Computes the slope of the regression line"""
    r = correlation(df, x, y)
    y_sd = np.std(df[y])
    x_sd = np.std(df[x])
    return r * y_sd / x_sd
    
def intercept(df, x, y):
    """Computes the intercept of the regression line"""
    x_mean = np.mean(df[x])
    y_mean = np.mean(df[y])
    return y_mean - slope(df, x, y)*x_mean

def fitted_values(df, x, y):
    """Return an array of the regressions estimates at all the x values"""
    a = slope(df, x, y)
    b = intercept(df, x, y)
    return a*df[x] + b

## Residual Demonstration using 2016 election dataset

In [ ]:
demographics = pd.read_csv('data/district_demographics2016.csv')
demographics.head(10)

In [ ]:
predict_voting = demographics[['Median Income', 'Percent voting for Clinton']].copy()
predict_voting['Fitted'] = fitted_values(demographics, 'Median Income', 'Percent voting for Clinton')

In [ ]:
plt.scatter(predict_voting['Median Income'], predict_voting['Percent voting for Clinton'], 
           label='Percent voting for Clinton')
plt.scatter(predict_voting['Median Income'], predict_voting['Fitted'], 
            label='Fitten Line')
plt.xlabel('Median Income')
plt.ylabel('Percent voting for Clinton')
plt.legend();

### 🧠 Description

This code creates a scatter plot comparing **actual** and **fitted** values from a regression model that predicts **voting behavior based on income**.


In [ ]:
predict_income = demographics[['College%', 'Median Income']].copy()
predict_income['Fitted'] = fitted_values(demographics, 'College%', 'Median Income')

In [ ]:
plt.scatter(predict_income['College%'], predict_income['Median Income'], 
           label='Median Income')
plt.scatter(predict_income['College%'], predict_income['Fitted'], 
            label='Fitted Line')
plt.xlabel('College%')
plt.ylabel('Median Income')
plt.legend();

### Demo 4: Residual

In [ ]:
demos = demographics.drop(['State', 'District', 'Percent voting for Clinton'], axis=1)
demos.head(5)

In [ ]:
def residuals(df, x, y):
    predictions = fitted_values(df, x, y)
    return df[y] - predictions

In [ ]:
demos['Fitted Value'] = fitted_values(demos, 'College%', 'Median Income')
demos['Residual'] = residuals(demos, 'College%', 'Median Income')
demos.head(5)

In [ ]:
plt.scatter(demos['College%'], demos['Median Income'], 
           label='Median Income')
plt.scatter(demos['College%'], demos['Fitted Value'], 
            label='Fitted Line')
plt.scatter(demos['College%'], demos['Residual'], 
            label='Residual')
plt.xlabel('College%')
plt.ylabel('Median Income')
plt.legend();

### 🧠 Description

This plot compares **actual**, **fitted**, and **residual** values from a regression predicting **median income** based on **college percentage**.  
It shows how well the model fits and highlights areas of larger error.


In [ ]:
def plot_residuals(df, x, y):
    df['Fitted Value'] = fitted_values(df, x, y)
    df['Residual'] = residuals(df, x, y)
    plt.scatter(df[x], df[y], label=y)
    plt.scatter(df[x], df['Fitted Value'], label='Fitted Line')
    plt.xlabel('College%')
    plt.ylabel('Median Income')
    plt.legend();

    df.plot.scatter(x=x, y='Residual')

In [ ]:
family_heights = pd.read_csv('data/family_heights.csv')
parents = (family_heights['father'] + family_heights['mother'])/2
heights = pd.DataFrame({
    'Parent Average': parents,
    'Child': family_heights['child']}
    )
plot_residuals(heights, 'Parent Average', 'Child')

## 🏠 Homework: Practice with Your Peers or Friends

## Demo 5: Dugongs Data

In [ ]:
dugong = pd.read_csv('data/dugong.csv')
dugong.head(5)

In [ ]:
dugong.plot.scatter(x='Length', y='Age')

In [ ]:
correlation(dugong, 'Length', 'Age')

In [ ]:
plot_residuals(dugong, 'Length', 'Age')

## Demo 6: US Women

In [ ]:
us_women = pd.read_csv('data/us_women.csv')
us_women.head(5)

In [ ]:
us_women.plot.scatter(x='height', y='ave weight');

In [ ]:
correlation(us_women, 'height', 'ave weight')

In [ ]:
plot_residuals(us_women, 'height', 'ave weight')

### Average of Residuals

In [ ]:
round(np.average(residuals(dugong, 'Length', 'Age')), 6)

In [ ]:
round(np.average(residuals(heights, 'Parent Average', 'Child')), 6)

In [ ]:
round(np.average(residuals(demographics, 'College%', 'Median Income')), 6)

In [ ]:
round(correlation(heights, 'Parent Average', 'Residual'), 6)

In [ ]:
round(correlation(heights, 'Fitted Value', 'Residual'), 6)

# See You Next Lecture!